In [11]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit, train_test_split

In [12]:
LABEL_LIST = [
    "O",
    "B-VOL", "I-VOL",
    "B-LOC", "I-LOC",
    "B-HAZ", "I-HAZ",
    "B-TIME", "I-TIME",
    "B-GOV", "I-GOV",
    "B-SOC", "I-SOC",
]
label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for i, label in enumerate(LABEL_LIST)}

In [13]:
print("Loading IndoBERT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

Loading IndoBERT Tokenizer...


In [14]:
OUTPUT_DIR = Path("../data/ner_distribution/data_distribution")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [15]:
def align_labels_with_tokens(text, annotations):
    """
    Converts Label Studio character offsets into BIO token labels.
    """
    # 1. Tokenize the text, asking for character offsets
    encoding = tokenizer(
        text, 
        truncation=True, 
        max_length=256, 
        return_offsets_mapping=True
    )
    offsets = encoding["offset_mapping"]
    
    # 2. Create an array of "O" for every character in the text
    char_labels = ["O"] * len(text)
    
    # Dictionary to map Label Studio names to your LABEL_LIST names
    LABEL_MAPPING = {
        "VOLCANO": "VOL",
        "LOCATION": "LOC",
        "HAZARD": "HAZ",
        "TIME": "TIME",
        "GOVERNMENT": "GOV",
        "SOCIAL ACTORS": "SOC"
    }
    
    # 3. Fill in the character labels based on Label Studio annotations
    for ann in annotations:
        start = ann["value"]["start"]
        end = ann["value"]["end"]
        raw_label = ann["value"]["labels"][0]
        
        # Translate "VOLCANO" -> "VOL"
        label = LABEL_MAPPING.get(raw_label, raw_label)
        
        # Assign the label to every character in the span
        for i in range(start, end):
            if i < len(char_labels):
                char_labels[i] = label

    # 4. Map the character labels to the subword tokens
    label_ids = []
    
    for i, (start_char, end_char) in enumerate(offsets):
        # Special tokens like [CLS] and [SEP] have start==end
        if start_char == end_char:
            label_ids.append(-100) # -100 is ignored by PyTorch
            continue
            
        # Get the label for the first character of this token
        current_char_label = char_labels[start_char]
        
        if current_char_label == "O":
            label_ids.append(label2id["O"])
        else:
            # FIXED LOGIC: Look back one character to decide if it's B- or I-
            if start_char == 0 or char_labels[start_char - 1] != current_char_label:
                label_ids.append(label2id[f"B-{current_char_label}"])
            else:
                label_ids.append(label2id[f"I-{current_char_label}"])

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": label_ids
    }

In [19]:
def main():
    INPUT_PATH = Path("../data/ner_distribution/annotated/annotated.json")
    print(f"Loading Label Studio export from {INPUT_PATH}...")
    
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        ls_data = json.load(f)

    processed_data = []
    
    print(f"Processing {len(ls_data)} annotated tasks...")
    for task in ls_data:
        text = task["data"]["text"]
        video_title = task["data"].get("video_title", "unknown_title")
        
        if not task.get("annotations") or len(task["annotations"]) == 0:
            continue
            
        results = task["annotations"][0]["result"]
        tokenized_record = align_labels_with_tokens(text, results)
        tokenized_record["video_title"] = video_title
        processed_data.append(tokenized_record)

    print(f"Successfully processed {len(processed_data)} annotated sentences.")

    # --- CHECK UNIQUE DOCUMENTS ---
    unique_docs = list(set([item["video_title"] for item in processed_data]))
    print(f"Found {len(unique_docs)} unique documents.")

    # --- SPLITTING THE DATA ---
    if len(unique_docs) >= 3:
        print("Using GroupShuffleSplit (Splitting by Document)...")
        video_titles = [item["video_title"] for item in processed_data]
        
        gss1 = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
        train_idx, temp_idx = next(gss1.split(processed_data, groups=video_titles))
        
        train_data = [processed_data[i] for i in train_idx]
        temp_data = [processed_data[i] for i in temp_idx]
        temp_video_titles = [video_titles[i] for i in temp_idx]
        
        gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
        val_idx, test_idx = next(gss2.split(temp_data, groups=temp_video_titles))
        
        val_data = [temp_data[i] for i in val_idx]
        test_data = [temp_data[i] for i in test_idx]

    else:
        print("Not enough unique documents for a 3-way group split. Using Random Split.")
        df = pd.DataFrame(processed_data)
        
        train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
        val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
        
        train_data = train_df.to_dict('records')
        val_data = val_df.to_dict('records')
        test_data = test_df.to_dict('records')

    # --- SAVING TO JSONL ---
    def save_jsonl(data, filename):
        filepath = OUTPUT_DIR / filename
        with open(filepath, "w", encoding="utf-8") as f:
            for item in data:
                # Remove video_title before saving (not needed for training)
                item.pop("video_title", None)
                f.write(json.dumps(item) + "\n")
        print(f"Saved {len(data)} examples to {filepath}")

    save_jsonl(train_data, "train.jsonl")
    save_jsonl(val_data, "val.jsonl")
    save_jsonl(test_data, "test.jsonl")

    # Save label maps for Phase 3 (Training)
    label_map_path = "../data/ner_distribution/data_distribution/label_map.json"
    with open(label_map_path, "w", encoding="utf-8") as f:
        json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)
    
    print(f" Saved label_map.json to {label_map_path}")
    print("Ready for IndoBERT fine-tuning.")

if __name__ == "__main__":
    main()

Loading Label Studio export from ..\data\ner_distribution\annotated\annotated.json...
Processing 64836 annotated tasks...
Successfully processed 64836 annotated sentences.
Found 325 unique documents.
Using GroupShuffleSplit (Splitting by Document)...
Saved 55101 examples to ..\data\ner_distribution\data_distribution\train.jsonl
Saved 3054 examples to ..\data\ner_distribution\data_distribution\val.jsonl
Saved 6681 examples to ..\data\ner_distribution\data_distribution\test.jsonl
 Saved label_map.json to ../data/ner_distribution/data_distribution/label_map.json
Ready for IndoBERT fine-tuning.
